# Parte 1 – Scraping de decretos de emergencia
Grupo 9 · Temporada: 1 de enero al 31 de mayo de 2022

In [1]:
import pandas as pd

In [2]:
import sys
!"{sys.executable}" -m ensurepip --upgrade
%pip install selenium requests beautifulsoup4 pandas

Looking in links: c:\Users\huarh\AppData\Local\Temp\tmpqefxpdsh
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests

respuesta = requests.get("https://www.gob.pe/robots.txt", headers={"User-Agent": "Mozilla/5.0"})
print(respuesta.text)

# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

base = ("https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
        "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent")
url = base + "&desde=01-01-2022&hasta=31-01-2022"

driver = webdriver.Chrome()
driver.get(url)
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "article")))

articulos = driver.find_elements(By.TAG_NAME, "article")
print(len(articulos), "articles en pantalla")
print(articulos[0].get_attribute("outerHTML"))

8 articles en pantalla
<article class="py-8 md:w-3/4" aria-labelledby="result-title-2240387-label"><h3 class="text-xs text-gray-650 mb-2 font-semibold">PCM - Presidencia del Consejo de Ministros</h3><div class="flex gap-2 md:gap-4 items-center mb-2 flex-wrap"><ol class="flex gap-2 font-semibold"><li><span>Normas y documentos legales</span></li><li class="flex gap-2">›<span>Decretos</span></li></ol></div><h4 id="result-title-2240387-label" class="text-xl font-semibold mb-2" aria-describedby="result-title-2240387-description"><a class="track-ga-click link-principal underline" data-origin="result-rule-2724291" href="/institucion/pcm/normas-legales/2724291-011-2022-pcm">Decreto Supremo N.° 011-2022-PCM</a></h4><h5 class="text-gray-725 mb-2 font-normal"></h5><div><span class="text-gray-900 font-bold">Publicado: 30 de enero de 2022</span><p id="result-title-2240387-description" class="mb-0">Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decre

In [5]:
import re

# 1) Número total que muestra la página ("N Resultados")
texto_pagina = driver.find_element(By.TAG_NAME, "body").text
m = re.search(r"(\d+)\s+[Rr]esultados?", texto_pagina)
print("Total según la página:", m.group(1) if m else "NO ENCONTRADO")

# 2) Datos de cada article
filas = []
for art in articulos:
    # todos los enlaces del article; el principal es el primero que tiene texto
    enlaces = art.find_elements(By.TAG_NAME, "a")
    enlace_el = next((a for a in enlaces if a.text.strip()), enlaces[0])
    lineas = [l.strip() for l in art.text.split("\n") if l.strip()]
    fecha = next((l for l in lineas if l.startswith("Publicado")), "")
    i = lineas.index(fecha) if fecha in lineas else -1
    titulo = lineas[i + 1] if 0 <= i < len(lineas) - 1 else ""
    filas.append({
        "numero": enlace_el.text.strip(),
        "fecha": fecha.replace("Publicado:", "").strip(),
        "titulo": titulo,
        "enlace": enlace_el.get_attribute("href"),
    })

# para revisar: texto y enlaces del primer article
print(articulos[0].text)
print([a.get_attribute("href") for a in articulos[0].find_elements(By.TAG_NAME, "a")])

Total según la página: 8
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 011-2022-PCM
Publicado: 30 de enero de 2022
Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la ...
Descargar archivo
['https://www.gob.pe/institucion/pcm/normas-legales/2724291-011-2022-pcm', 'https://cdn.www.gob.pe/uploads/document/file/2794407/Decreto%20Supremo%20N%C2%B0011-2022-PCM.pdf']


In [6]:
import time
from selenium.common.exceptions import TimeoutException

meses = [
    ("enero",   "01-01-2022", "31-01-2022"),
    ("febrero", "01-02-2022", "28-02-2022"),
    ("marzo",   "01-03-2022", "31-03-2022"),
    ("abril",   "01-04-2022", "30-04-2022"),
    ("mayo",    "01-05-2022", "31-05-2022"),
]

def extraer_article(art):
    lineas = [l.strip() for l in art.text.split("\n") if l.strip()]
    fecha = next((l for l in lineas if l.startswith("Publicado")), "")
    i = lineas.index(fecha) if fecha in lineas else -1
    hrefs = [a.get_attribute("href") for a in art.find_elements(By.TAG_NAME, "a")]
    enlace = next((h for h in hrefs if h and "/normas-legales/" in h), hrefs[0] if hrefs else "")
    return {
        "numero": lineas[i - 1] if i > 0 else "",
        "fecha": fecha.replace("Publicado:", "").strip(),
        "titulo": lineas[i + 1] if 0 <= i < len(lineas) - 1 else "",
        "enlace": enlace,
    }

todas = []
verificacion = []
for mes, desde, hasta in meses:
    driver.get(base + f"&desde={desde}&hasta={hasta}")
    try:
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "article")))
        arts = driver.find_elements(By.TAG_NAME, "article")
    except TimeoutException:
        arts = []          # mes sin resultados
    texto = driver.find_element(By.TAG_NAME, "body").text
    m = re.search(r"(\d+)\s+[Rr]esultados?", texto)
    total = int(m.group(1)) if m else 0
    for art in arts:
        fila = extraer_article(art)
        fila["mes"] = mes
        todas.append(fila)
    verificacion.append({"mes": mes, "resultados_totales": total, "resultados_extraidos": len(arts)})
    time.sleep(2)          # pausa entre páginas

tabla_verificacion = pd.DataFrame(verificacion)
tabla_verificacion

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=154.0.8037.95)
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6c1bf9d55+5a55]
	chromedriver!(No symbol) [0x7ff6c1b1f540]
	chromedriver!(No symbol) [0x7ff6c19445fd]
	chromedriver!(No symbol) [0x7ff6c191b5d2]
	chromedriver!(No symbol) [0x7ff6c19d07f6]
	chromedriver!(No symbol) [0x7ff6c19ef5bd]
	chromedriver!(No symbol) [0x7ff6c1991a12]
	chromedriver!(No symbol) [0x7ff6c1992823]
	chromedriver!GetHandleVerifier [0x7ff6c20104b1+41c1b1]
	chromedriver!GetHandleVerifier [0x7ff6c203e26b+449f6b]
	chromedriver!GetHandleVerifier [0x7ff6c203274e+43e44e]
	chromedriver!GetHandleVerifier [0x7ff6c1cf32fe+feffe]
	chromedriver!(No symbol) [0x7ff6c1b2cb55]
	chromedriver!(No symbol) [0x7ff6c1b27f64]
	chromedriver!(No symbol) [0x7ff6c1b280f4]
	chromedriver!(No symbol) [0x7ff6c1b12e4c]
	KERNEL32!BaseThreadInitThunk [0x7ffcfac7cd87+17]
	ntdll!RtlUserThreadStart [0x7ffc

In [ ]:
driver.quit()   # ya no necesitamos el navegador

df = pd.DataFrame(todas)
print("Filas extraídas:", len(df))

df = df.drop_duplicates(subset="enlace")
print("Después de quitar repetidas:", len(df))

es_pcm = df["enlace"].str.contains("/institucion/pcm/")
print("Normas de otras instituciones quitadas:", (~es_pcm).sum())
print(df.loc[~es_pcm, ["numero", "enlace"]])

df = df[es_pcm].reset_index(drop=True)
print("Normas de la PCM que quedan:", len(df))

Filas extraídas: 29
Después de quitar repetidas: 29
Normas de otras instituciones quitadas: 0
Empty DataFrame
Columns: [numero, enlace]
Index: []
Normas de la PCM que quedan: 29


En nuestra temporada (enero a mayo de 2022) no se coló ninguna norma de otra institución: las 29 tienen `/institucion/pcm/` en el enlace, así que quitamos 0. Tampoco hubo repetidas.

In [ ]:
from bs4 import BeautifulSoup

titulos_completos = []
for enlace in df["enlace"]:
    r = requests.get(enlace, headers={"User-Agent": "Mozilla/5.0"})
    sopa = BeautifulSoup(r.text, "html.parser")
    div = sopa.find("div", class_="description")
    titulos_completos.append(div.get_text(" ", strip=True) if div else "")
    time.sleep(1)   # pausa entre páginas

df["titulo"] = titulos_completos
print("Títulos vacíos:", (df["titulo"] == "").sum())
df[["numero", "fecha", "titulo"]].head()

Títulos vacíos: 0


,numero,fecha,titulo
0,Decreto Supremo N.° 011-2022-PCM,30 de enero de 2022,Decreto Supremo que modifica el numeral 14.2 d...
1,Decreto Supremo N.° 010-2022-PCM,29 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...
2,Decreto Supremo N.° 009-2022-PCM,28 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...
3,Decreto Supremo N.° 004-2022-PCM,25 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...
4,Decreto Supremo N.° 007-2022-PCM,18 de enero de 2022,Decreto Supremo que prorroga el Estado de Emer...


In [ ]:
def clasificar_tipo(t):
    if "prórroga" in t or "prorroga" in t:
        return "prorroga"
    elif "declara" in t:
        return "declara"
    else:
        return "otro"

def clasificar_motivo(t):
    if "peligro inminente" in t:
        return "peligro inminente"
    elif "impacto de daños" in t:
        return "impacto de daños"
    else:
        return "otro"

minus = df["titulo"].str.lower()
df["es_lluvia"] = minus.apply(lambda t: "lluvia" in t or "precipitaciones" in t)
df["tipo"] = minus.apply(clasificar_tipo)
df["motivo"] = minus.apply(clasificar_motivo)

print(df["es_lluvia"].value_counts(), "\n")
print(df["tipo"].value_counts(), "\n")
print(df["motivo"].value_counts(), "\n")

# filas que quedaron como "otro", para revisarlas a mano
pd.set_option("display.max_colwidth", None)
df.loc[(df["tipo"] == "otro") | (df["motivo"] == "otro"), ["numero", "tipo", "motivo", "titulo"]]

es_lluvia
False    28
True      1
Name: count, dtype: int64 

tipo
prorroga    17
declara     11
otro         1
Name: count, dtype: int64 

motivo
otro                22
impacto de daños     7
Name: count, dtype: int64 



,numero,tipo,motivo,titulo
0,Decreto Supremo N.° 011-2022-PCM,declara,otro,Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social Decreto Supremo N°011-2022-PCM PDF 1.3 MB Descargar
1,Decreto Supremo N.° 010-2022-PCM,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 13 de febrero de 2022 . Decreto Supremo N°010-2022-PCM PDF 270.7 KB Descargar"
3,Decreto Supremo N.° 004-2022-PCM,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Puerto Inca, Tournavista, Yuyapichis, Codo del Pozuzo y Honoria de la provincia de Puerto Inca del departamento de Huánuco, en los distritos de Constitución, Palcazú y Puerto Bermúdez de la provincia de Oxapampa del departamento de Pasco y en los distritos de Raimondi, Sepahua y Tahuanía de la provincia de Atalaya del departamento de Ucayali DS N° 004-2022-PCM.pdf PDF 910.1 KB Descargar"
4,Decreto Supremo N.° 007-2022-PCM,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto DS N° 007-2022-PCM.pdf PDF 791.4 KB Descargar
5,Decreto Supremo N.° 006-2022-PCM,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata, en los distritos de Fitzcarrald, Manu, Madre de Dios y Huepetuhe de la provincia de Manu y en los distritos de Iñapari, Iberia y Tahuamanu de la provincia de Tahuamanu del departamento de Madre de Dios DS N° 006-2022-PCM.pdf PDF 918.2 KB Descargar"
6,Decreto Supremo N.° 005-2022-PCM,declara,otro,Decreto Supremo que modifica el Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 30 de enero de 2022 . Decreto Supremo N°005-2022-PCM PDF 827.1 KB Descargar
7,Decreto Supremo N.° 002-2022-PCM,declara,otro,"Decreto Supremo que modifica el numeral 8.1 del artículo 8 del Decreto Supremo N° 184-2020-PCM, Decreto Supremo quedeclara Estado de Emergencia Nacional por las graves circunstancias que afectanla vida de las personas a consecuenciade la COVID-19 y establece las medidasque debe seguir la ciudadanía en la nueva convivencia social. Decreto Supremo 002-2022-PCM.pdf PDF 166 KB Descargar"
8,Decreto Supremo N.° 016-2022-PCM,declara,otro,Decreto Supremo que declara Estado de Emergencia Nacional por las circunstancias que afectan la vida y salud de las personas como consecuencia de la COVID-19 y establece nuevas medidas para el restablecimiento de la convivencia social Decreto Supremo N016-2022-PCM.pdf PDF 196 KB Descargar
9,Decreto Supremo N.° 015-2022-PCM,declara,otro,"Decreto Supremo que modifica el Decreto Supremo N° 184-2020-PCM, Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social desde el 14 de febrero de 2022. Decreto Supremo N°015-2022-PCM PDF 265.3 KB Descargar"
11,Decreto Supremo N.° 013-2022-PCM,prorroga,otro,"Prórroga del Estado de Emergencia en distritos de las provincias de Huanta y La Mar del departamento de Ayacucho; de las provincias de Tayacaja y Churcampa del departamento de Huancavelica; de la provincia de La Convención del departamento de Cusco; de las provincias de Satipo, Concep

In [ ]:
for i, fila in df.iterrows():
    print(i, fila["numero"], "|", fila["tipo"], "|", fila["motivo"])
    print(fila["titulo"][:400])
    print()

0 Decreto Supremo N.° 011-2022-PCM | declara | otro
Decreto Supremo que modifica el numeral 14.2 del artículo 14 del Decreto Supremo N° 184-2020-PCM Decreto Supremo que declara Estado de Emergencia Nacional por las graves circunstancias que afectan la vida de las personas a consecuencia de la COVID-19 y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social Decreto Supremo N°011-2022-PCM PDF 1.3 MB Descargar

1 Decreto Supremo N.° 010-2022-PCM | prorroga | otro
Decreto Supremo que prorroga el Estado de Emergencia Nacional, modifica el Decreto Supremo N°184-2020-PCM y establece las medidas que debe seguir la ciudadanía en la nueva convivencia social hasta el 13 de febrero de 2022 . Decreto Supremo N°010-2022-PCM PDF 270.7 KB Descargar

2 Decreto Supremo N.° 009-2022-PCM | prorroga | impacto de daños
Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Cajamarca, Loreto y San Martín, p

In [ ]:
print(df.loc[df["tipo"] == "otro", ["numero", "titulo"]].to_string())
print()
print(df.loc[df["es_lluvia"], ["numero", "tipo", "motivo", "titulo"]].to_string())

                                                              numero                                                                                                                                                                                                                                                                                                                                    titulo
17  Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP  Dejan sin efecto la Res. N° 002-2021-PCM/ SGP y adecúan de manera progresiva el horario de atención al público de los Centros MAC de la Plataforma MAC durante el estado de emergencia sanitaria por el COVID-19. Esta norma pertenece al compendio Resoluciones de Secretaría de Gestión Pública RSGP_003_2022_PCM PDF 122 KB Descargar

                              numero     tipo            motivo                                                                                                                                          

### Revisión de las filas clasificadas como "otro"
- La mayoría de normas con `motivo = "otro"` están **bien clasificadas**: no son por lluvias. Son prórrogas o modificaciones de la emergencia nacional por COVID-19 (D.S. 184-2020-PCM) y emergencias por seguridad en Putumayo, Puerto Inca y Madre de Dios. No hablan de peligro inminente ni de impacto de daños.
- El D.S. 009-2022-PCM tiene `motivo = "impacto de daños"`, pero por un **movimiento sísmico**. Está bien que `es_lluvia` sea False: la frase coincide, pero la causa no es la lluvia.
- Fila con `tipo = "otro"`: es la Resolución de Secretaría de Gestión Pública N.° 003-2022-PCM/SGP, sobre el horario de los Centros MAC por el COVID-19. No declara ni prorroga un estado de emergencia, así que "otro" es correcto.

In [ ]:
lluvias = df[df["es_lluvia"]].reset_index(drop=True)
print("Normas de lluvias:", len(lluvias))
lluvias.head(5)

Normas de lluvias: 1


,numero,fecha,titulo,enlace,mes,es_lluvia,tipo,motivo
0,Decreto Supremo N.° 032-2022-PCM,1 de abril de 2022,"Decreto Supremo que declara el Estado de Emergencia en algunos distritos de varias provincias de los departamentos de Amazonas, Ayacucho y Piura, por impacto de daños a consecuencia de intensas precipitaciones pluviales DS N° 032-2022-PCM.pdf PDF 2.6 MB Descargar",https://www.gob.pe/institucion/pcm/normas-legales/2929443-032-2022-pcm,abril,True,declara,impacto de daños


In [ ]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]

def deps_en(titulo):
    # título original (con mayúsculas) y palabra completa (\b)
    # cada departamento entra UNA sola vez a la lista, aunque se repita en el título
    return [d for d in departamentos if re.search(r"\b" + d + r"\b", titulo)]

lluvias["departamentos"] = lluvias["titulo"].apply(deps_en)
print(lluvias[["numero", "tipo", "departamentos"]].to_string())

# Prueba del problema de "Ica" con un ejemplo
ejemplo = "provincia de Huancavelica del departamento de Huancavelica"
print("Búsqueda en minúsculas:", [d for d in departamentos if d.lower() in ejemplo.lower()])
print("Búsqueda correcta:     ", deps_en(ejemplo))

                             numero     tipo                departamentos
0  Decreto Supremo N.° 032-2022-PCM  declara  [Amazonas, Ayacucho, Piura]
Búsqueda en minúsculas: ['Huancavelica', 'Ica']
Búsqueda correcta:      ['Huancavelica']


In [ ]:
lluvias[["numero", "fecha", "titulo", "tipo", "motivo", "enlace"]].to_csv(
    "datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")

filas_dep = []
for d in departamentos:
    menciona = lluvias["departamentos"].apply(lambda lista: d in lista)
    filas_dep.append({
        "departamento": d,
        "declaratorias": int((menciona & (lluvias["tipo"] == "declara")).sum()),
        "prorrogas": int((menciona & (lluvias["tipo"] == "prorroga")).sum()),
    })
por_dep = pd.DataFrame(filas_dep)
por_dep.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")
print("Filas:", len(por_dep))
por_dep[(por_dep["declaratorias"] + por_dep["prorrogas"]) > 0]

Filas: 25


,departamento,declaratorias,prorrogas
0,Amazonas,1,0
4,Ayacucho,1,0
19,Piura,1,0


### ¿Cómo evitamos contar mal los departamentos?
En nuestra temporada solo hay un decreto de lluvias, el D.S. 032-2022-PCM, que menciona **Amazonas, Ayacucho y Piura**. Lo encontramos buscando cada departamento en el **título original** (con mayúsculas) y como **palabra completa** con `re.search(r"\b" + departamento + r"\b", titulo)`.

En nuestros datos no aparece Huancavelica, así que probamos el problema con un ejemplo. En el texto "provincia de Huancavelica del departamento de Huancavelica", la búsqueda en minúsculas encuentra `['Huancavelica', 'Ica']` (porque "ica" está dentro de "huancavelica"), mientras que nuestra búsqueda encuentra solo `['Huancavelica']`.

Además, cada departamento entra **una sola vez** a la lista de cada decreto, porque recorremos la lista de departamentos y no las palabras del título. Así, un texto como "provincia de Ica del departamento de Ica" cuenta Ica una sola vez.